# Prepare Basic Attribute Dataset

Build a lightweight training-ready hairstyle attribute dataset from the reviewed kept asset bank using the most learnable first-step fields: `length` and `curl`.

In [7]:
from pathlib import Path
import sys

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'backend').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate project root.')

PROJECT_ROOT = find_project_root()
BACKEND_ROOT = PROJECT_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.append(str(BACKEND_ROOT))

PROJECT_ROOT

WindowsPath('D:/Projects/Personal Projects/Hairstyle Recommender Live Tryon')

In [8]:
import json
import pandas as pd

from app.ml.datasets import (
    build_attribute_records_from_reviewed_assets,
    build_label_vocab_for_fields,
    train_val_split,
    write_jsonl_manifest,
)

REVIEWED_ROOT = BACKEND_ROOT / 'data' / 'processed' / 'celeba_hair_rich_assets' / 'reviewed'
LABELED_KEPT_JSONL = REVIEWED_ROOT / 'kept_assets_labeled.jsonl'

DATASET_ROOT = BACKEND_ROOT / 'data' / 'datasets' / 'reviewed_hairstyle_basic'
TRAIN_PATH = DATASET_ROOT / 'train.jsonl'
VAL_PATH = DATASET_ROOT / 'val.jsonl'
VOCAB_PATH = DATASET_ROOT / 'label_vocab.json'
SUMMARY_PATH = DATASET_ROOT / 'summary.json'

CORE_FIELDS = ('length', 'curl')
TRAIN_RATIO = 0.8
SEED = 42

DATASET_ROOT

WindowsPath('D:/Projects/Personal Projects/Hairstyle Recommender Live Tryon/backend/data/datasets/reviewed_hairstyle_basic')

In [9]:
records = build_attribute_records_from_reviewed_assets(
    LABELED_KEPT_JSONL,
    required_fields=CORE_FIELDS,
)

print('Basic records:', len(records))
records[0] if records else None

Basic records: 84


{'image_path': 'D:\\Projects\\Personal Projects\\Hairstyle Recommender Live Tryon\\backend\\data\\processed\\celeba_hair_rich_assets\\images\\celeba_hair_000002.png',
 'labels': {'length': 'long', 'curl': 'wavy'},
 'source_id': 'celeba_hair_000002',
 'source_dataset': 'CelebA-reviewed-kept'}

In [10]:
train_records, val_records = train_val_split(records, train_ratio=TRAIN_RATIO, seed=SEED)
label_vocab = build_label_vocab_for_fields(records, CORE_FIELDS)

DATASET_ROOT.mkdir(parents=True, exist_ok=True)
write_jsonl_manifest(train_records, TRAIN_PATH)
write_jsonl_manifest(val_records, VAL_PATH)
VOCAB_PATH.write_text(json.dumps(label_vocab, indent=2), encoding='utf-8')

summary = {
    'core_fields': list(CORE_FIELDS),
    'total_records': len(records),
    'train_records': len(train_records),
    'val_records': len(val_records),
    'train_ratio': TRAIN_RATIO,
    'field_value_counts': {
        field: pd.Series([record['labels'][field] for record in records]).value_counts().to_dict()
        for field in CORE_FIELDS
    },
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print('Wrote:', TRAIN_PATH)
print('Wrote:', VAL_PATH)
print('Wrote:', VOCAB_PATH)
print('Wrote:', SUMMARY_PATH)

Wrote: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\data\datasets\reviewed_hairstyle_basic\train.jsonl
Wrote: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\data\datasets\reviewed_hairstyle_basic\val.jsonl
Wrote: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\data\datasets\reviewed_hairstyle_basic\label_vocab.json
Wrote: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\data\datasets\reviewed_hairstyle_basic\summary.json


In [11]:
pd.Series(summary)

core_fields                                              [length, curl]
total_records                                                        84
train_records                                                        67
val_records                                                          17
train_ratio                                                         0.8
field_value_counts    {'length': {'long': 44, 'short': 35, 'medium':...
dtype: object

In [12]:
for field in CORE_FIELDS:
    print('\n', field)
    display(pd.Series(summary['field_value_counts'][field], name='count').sort_values(ascending=False))


 length


long      44
short     35
medium     5
Name: count, dtype: int64


 curl


straight    45
wavy        37
curly        2
Name: count, dtype: int64